<a href="https://colab.research.google.com/github/mirfanmbrk/Komputasi-Numerik/blob/main/SPL_3_Variabel_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Penyelesaian SPL Tiga Variabel dengan Eliminasi Gauss

Program ini menyelesaikan sistem persamaan linear dengan variabel t, u, dan v:

    a11·t + a12·u + a13·v = b1
    a21·t + a22·u + a23·v = b2
    a31·t + a32·u + a33·v = b3

**Metode:** Eliminasi Gauss, terdiri dari dua tahap:
1. **Eliminasi maju**: matriks augmented diubah menjadi matriks segitiga atas.
2. **Substitusi balik**: nilai v, lalu u, lalu t dihitung dari baris bawah ke atas.

Program juga menangani pivot nol (tukar baris) serta mendeteksi SPL yang
tidak memiliki solusi dan SPL yang memiliki tak hingga banyak solusi.

In [8]:
# CELL 2 - Pengaturan program
# Eliminasi Gauss ditulis sendiri dengan Python murni, jadi tidak perlu import library.

NAMA_VARIABEL = ["t", "u", "v"]   # nama tiga variabel pada SPL
EPSILON = 1e-9                    # toleransi: angka dengan |nilai| <= EPSILON dianggap nol

print("Pengaturan selesai.")
print("Variabel SPL  :", NAMA_VARIABEL)
print("Toleransi nol :", EPSILON)

Pengaturan selesai.
Variabel SPL  : ['t', 'u', 'v']
Toleransi nol : 1e-09


### Penjelasan Cell 2: Pengaturan Program

**Tujuan cell**
Menyiapkan dua pengaturan yang dipakai di seluruh program. Cell ini tidak memuat `import` karena Eliminasi Gauss ditulis sendiri dengan Python murni.

**Fungsi variabel**
| Variabel | Fungsi |
|---|---|
| `NAMA_VARIABEL` | Daftar (list) berisi nama variabel `t`, `u`, `v`. Dipakai agar hasil tercetak dengan nama yang benar. |
| `EPSILON` | Batas toleransi `1e-9` (= 0,000000001). Angka yang nilai mutlaknya lebih kecil atau sama dengan batas ini dianggap nol. |

**Mengapa perlu EPSILON?** Komputer menyimpan bilangan desimal secara pendekatan, sehingga hasil hitungan yang seharusnya tepat 0 bisa menjadi 0,00000000000000001. Karena itu angka diuji dengan `abs(nilai) > EPSILON`, bukan dengan `nilai == 0`.

**Fungsi tiap baris kode**
- Baris komentar (`#`): penjelasan untuk pembaca, tidak dijalankan.
- `NAMA_VARIABEL = ["t", "u", "v"]`: membuat list tiga nama variabel.
- `EPSILON = 1e-9`: menetapkan toleransi nol.
- Tiga perintah `print(...)`: menampilkan bahwa pengaturan sudah siap beserta isinya.

**Hubungan dengan Eliminasi Gauss**
`EPSILON` dipakai saat mencari pivot yang tidak nol (Cell 5) dan saat memeriksa baris berbentuk `0 0 0 | c` (Cell 6).

**Hasil yang diharapkan**
```
Pengaturan selesai.
Variabel SPL  : ['t', 'u', 'v']
Toleransi nol : 1e-09
```

In [9]:
# CELL 3 - Input koefisien dan konstanta
# Setiap baris A mewakili satu persamaan: [koefisien t, koefisien u, koefisien v]
A = [
    [ 2, 1, 3],    #  2t +  u + 3v =  1
    [ 4, 3, 1],    #  4t + 3u +  v = 13
    [-2, 5, 2],    # -2t + 5u + 2v = -5
]
b = [1, 13, -5]    # konstanta di ruas kanan tiap persamaan

n = len(A)         # jumlah persamaan (sekaligus jumlah variabel)

# Pemeriksaan bentuk data agar tidak terjadi kesalahan pada cell berikutnya
assert n == 3 and all(len(baris) == 3 for baris in A) and len(b) == 3, \
    "Data harus berupa SPL 3 persamaan dengan 3 variabel."

# Menampilkan SPL dalam bentuk persamaan agar mudah diperiksa
print("SPL yang akan diselesaikan:")
for i in range(n):
    ruas_kiri = " + ".join(f"({A[i][j]}){NAMA_VARIABEL[j]}" for j in range(n))
    print(f"  Persamaan {i+1}: {ruas_kiri} = {b[i]}")

SPL yang akan diselesaikan:
  Persamaan 1: (2)t + (1)u + (3)v = 1
  Persamaan 2: (4)t + (3)u + (1)v = 13
  Persamaan 3: (-2)t + (5)u + (2)v = -5


### Penjelasan Cell 3: Input Koefisien dan Konstanta

**Tujuan cell**
Memasukkan data SPL yang akan diselesaikan, lalu menampilkannya agar bisa dicek sebelum diproses.

**Fungsi variabel**
| Variabel | Fungsi |
|---|---|
| `A` | Matriks koefisien 3×3. Baris ke-i adalah persamaan ke-i, kolom ke-j adalah koefisien variabel ke-j (t, u, v). |
| `b` | Daftar konstanta ruas kanan, satu untuk tiap persamaan. |
| `n` | Jumlah persamaan, dihitung dengan `len(A)`. Dipakai sebagai batas perulangan di cell berikutnya. |
| `i`, `j` | Penghitung perulangan: `i` untuk nomor persamaan, `j` untuk nomor variabel. |
| `ruas_kiri` | Teks ruas kiri satu persamaan, misalnya `(2)t + (1)u + (3)v`. |

**Fungsi tiap baris kode**
- `A = [[...], [...], [...]]`: menyimpan 9 koefisien. Komentar di sampingnya menunjukkan bentuk persamaan aslinya.
- `b = [1, 13, -5]`: menyimpan 3 konstanta.
- `n = len(A)`: menghitung banyak baris `A`, hasilnya 3.
- `assert ...`: pemeriksaan data. `all(len(baris) == 3 for baris in A)` bernilai benar jika setiap baris punya tepat 3 koefisien. Jika syarat tidak terpenuhi, program berhenti dengan pesan kesalahan.
- `for i in range(n):`: mengulang untuk persamaan 1 sampai 3.
- `" + ".join(...)`: menyambung tiga suku (koefisien dan nama variabel) dengan tanda `+`.
- `print(f"...")`: mencetak satu persamaan lengkap.

**Hubungan dengan Eliminasi Gauss**
Ini adalah data awal metode. `A` dan `b` dipakai membentuk matriks augmented (Cell 4) dan **tidak diubah** selama eliminasi, sehingga bisa dipakai untuk validasi di Cell 8.

**Hasil yang diharapkan**
```
SPL yang akan diselesaikan:
  Persamaan 1: (2)t + (1)u + (3)v = 1
  Persamaan 2: (4)t + (3)u + (1)v = 13
  Persamaan 3: (-2)t + (5)u + (2)v = -5
```

In [10]:
# CELL 4 - Pembentukan matriks augmented

def tampil_matriks(M, judul=""):
    """Mencetak matriks augmented dengan garis pemisah sebelum kolom konstanta."""
    if judul:
        print(judul)
    for baris in M:
        # baris[:n] = kolom koefisien; +0.0 menghindari tampilan -0.000
        koefisien = "  ".join(f"{nilai + 0.0:9.3f}" for nilai in baris[:n])
        print(f"  [ {koefisien} | {baris[n] + 0.0:9.3f} ]")   # baris[n] = kolom konstanta
    print()

# Setiap baris M = koefisien persamaan + konstantanya; dijadikan float agar pembagian akurat
M = [[float(A[i][j]) for j in range(n)] + [float(b[i])] for i in range(n)]

tampil_matriks(M, "Matriks augmented awal:")

Matriks augmented awal:
  [     2.000      1.000      3.000 |     1.000 ]
  [     4.000      3.000      1.000 |    13.000 ]
  [    -2.000      5.000      2.000 |    -5.000 ]



### Penjelasan Cell 4: Pembentukan Matriks Augmented

**Tujuan cell**
Menggabungkan koefisien (`A`) dan konstanta (`b`) menjadi satu matriks 3×4 yang disebut **matriks augmented**, serta menyiapkan fungsi untuk mencetaknya.

**Fungsi variabel**
| Variabel | Fungsi |
|---|---|
| `M` | Matriks augmented (3 baris × 4 kolom). Kolom 1–3 adalah koefisien, kolom 4 adalah konstanta. **Matriks inilah yang diubah selama eliminasi.** |
| `tampil_matriks` | Fungsi pencetak matriks yang dipakai ulang di Cell 4 dan Cell 5. |
| `M` (parameter fungsi), `judul` | Matriks yang akan dicetak dan judul opsional di atasnya. |
| `baris`, `nilai`, `koefisien` | Satu baris matriks, satu angka dalam baris, dan teks tiga koefisien yang sudah diformat. |

**Fungsi tiap baris kode**
- `def tampil_matriks(M, judul=""):`: mendefinisikan fungsi. Fungsi baru dijalankan saat dipanggil.
- `if judul: print(judul)`: mencetak judul hanya jika judul diberikan.
- `for baris in M:`: mengulang untuk setiap baris matriks.
- `baris[:n]`: tiga elemen pertama (koefisien). `baris[n]`: elemen keempat (konstanta).
- `{nilai + 0.0:9.3f}`: angka dicetak selebar 9 karakter dengan 3 angka desimal. Penambahan `+ 0.0` mengubah `-0.0` menjadi `0.0` agar tidak tampil `-0.000`.
- `print()` di akhir fungsi: memberi baris kosong sebagai pemisah.
- `M = [[float(A[i][j]) for j in range(n)] + [float(b[i])] for i in range(n)]`: untuk setiap persamaan i, ambil tiga koefisien lalu tambahkan konstantanya di ujung. Semua diubah ke `float` agar hasil pembagian di Cell 5 berupa desimal.
- `tampil_matriks(M, "...")`: memanggil fungsi untuk mencetak matriks awal.

**Hubungan dengan Eliminasi Gauss**
Ini langkah "membentuk matriks augmented". Semua operasi baris elementer dilakukan pada `M`.

**Hasil yang diharapkan** (jumlah spasi bisa sedikit berbeda)
```
Matriks augmented awal:
  [     2.000      1.000      3.000 |     1.000 ]
  [     4.000      3.000      1.000 |    13.000 ]
  [    -2.000      5.000      2.000 |    -5.000 ]
```

In [11]:
# CELL 5 - Proses eliminasi Gauss (eliminasi maju)

r = 0   # r = indeks baris yang sedang dijadikan baris pivot (mulai dari 0)
print("=== ELIMINASI MAJU ===\n")

for j in range(n):                       # j = kolom yang sedang dieliminasi
    # 1) Cari baris p (dari r ke bawah) yang elemen kolom j-nya bukan nol
    p = None
    for i in range(r, n):
        if abs(M[i][j]) > EPSILON:
            p = i
            break

    # 2) Jika tidak ada pivot pada kolom ini, lewati kolom
    if p is None:
        print(f"Kolom {j+1} ({NAMA_VARIABEL[j]}): tidak ada pivot, kolom dilewati.\n")
        continue

    # 3) Jika posisi pivot bernilai nol, tukar baris
    if p != r:
        M[r], M[p] = M[p], M[r]
        tampil_matriks(M, f"Pivot nol -> tukar R{r+1} dengan R{p+1}:")

    print(f"Kolom {j+1} ({NAMA_VARIABEL[j]}): pivot = {M[r][j]:.3f} pada R{r+1}")

    # 4) Nolkan elemen kolom j di semua baris di bawah baris pivot
    for i in range(r + 1, n):
        m = M[i][j] / M[r][j]            # pengali
        for k in range(j, n + 1):
            M[i][k] = M[i][k] - m * M[r][k]
        M[i][j] = 0.0                    # elemen yang dinolkan dipastikan tepat 0
        print(f"  R{i+1} <- R{i+1} - ({m:.4g}) x R{r+1}")

    tampil_matriks(M, "Hasil operasi pada kolom ini:")
    r += 1                               # pindah ke baris pivot berikutnya

rank = r   # jumlah baris pivot yang berhasil dibentuk
print(f"Eliminasi maju selesai. Jumlah pivot (rank) = {rank}\n")
tampil_matriks(M, "Matriks setelah eliminasi:")

=== ELIMINASI MAJU ===

Kolom 1 (t): pivot = 2.000 pada R1
  R2 <- R2 - (2) x R1
  R3 <- R3 - (-1) x R1
Hasil operasi pada kolom ini:
  [     2.000      1.000      3.000 |     1.000 ]
  [     0.000      1.000     -5.000 |    11.000 ]
  [     0.000      6.000      5.000 |    -4.000 ]

Kolom 2 (u): pivot = 1.000 pada R2
  R3 <- R3 - (6) x R2
Hasil operasi pada kolom ini:
  [     2.000      1.000      3.000 |     1.000 ]
  [     0.000      1.000     -5.000 |    11.000 ]
  [     0.000      0.000     35.000 |   -70.000 ]

Kolom 3 (v): pivot = 35.000 pada R3
Hasil operasi pada kolom ini:
  [     2.000      1.000      3.000 |     1.000 ]
  [     0.000      1.000     -5.000 |    11.000 ]
  [     0.000      0.000     35.000 |   -70.000 ]

Eliminasi maju selesai. Jumlah pivot (rank) = 3

Matriks setelah eliminasi:
  [     2.000      1.000      3.000 |     1.000 ]
  [     0.000      1.000     -5.000 |    11.000 ]
  [     0.000      0.000     35.000 |   -70.000 ]



### Penjelasan Cell 5: Proses Eliminasi Gauss (Eliminasi Maju)

**Tujuan cell**
Mengubah matriks augmented `M` menjadi **matriks segitiga atas**, yaitu semua elemen di bawah diagonal utama bernilai 0, dengan operasi baris elementer.

**Fungsi variabel**
| Variabel | Fungsi |
|---|---|
| `r` | Nomor baris pivot saat ini (dimulai dari 0). Naik 1 setiap pivot berhasil ditemukan. |
| `j` | Nomor kolom yang sedang dieliminasi (0 = t, 1 = u, 2 = v). |
| `p` | Nomor baris yang dipilih menjadi pivot. Bernilai `None` jika tidak ada. |
| `i` | Nomor baris yang sedang diperiksa atau diubah. |
| `m` | **Pengali**: (elemen yang akan dinolkan) ÷ pivot. |
| `k` | Nomor kolom saat satu baris dikurangi kelipatan baris pivot. |
| `rank` | Jumlah pivot yang berhasil dibentuk. Dipakai di Cell 6 untuk menentukan jenis solusi. |

**Fungsi tiap bagian kode**
- `r = 0`: baris pivot pertama adalah baris pertama.
- `for j in range(n):`: memproses kolom satu per satu dari kiri.
- **Bagian 1, mencari pivot:** `for i in range(r, n)` memeriksa baris dari `r` ke bawah. Baris pertama dengan `abs(M[i][j]) > EPSILON` (bukan nol) disimpan di `p`, lalu `break` menghentikan pencarian.
- **Bagian 2:** jika `p is None` (Jika tidak ditemukan elemen pivot yang memenuhi toleransi pada baris aktif hingga baris terakhir, kolom tersebut dilewati), seluruh kolom bernilai nol sehingga kolom dilewati (`continue`). Ini bagian dari penanganan SPL tak hingga solusi.
- **Bagian 3, tukar baris:** jika `p != r`, pivot di posisi `r` bernilai nol, sehingga baris `r` dan `p` ditukar dengan `M[r], M[p] = M[p], M[r]`. Ini penanganan **pivot nol**.
- **Bagian 4, eliminasi:** untuk setiap baris di bawah pivot, `m = M[i][j] / M[r][j]` menghitung pengali, lalu `M[i][k] = M[i][k] - m * M[r][k]` mengurangi baris itu dengan `m` kali baris pivot, mulai dari kolom `j` sampai kolom konstanta. `M[i][j] = 0.0` memastikan elemen yang dinolkan tepat 0.
- `print(...)` dan `tampil_matriks(...)`: menampilkan setiap operasi dan matriks hasilnya agar tahapan terlihat.
- `r += 1`: pindah ke baris pivot berikutnya.
- `rank = r`: menyimpan jumlah pivot setelah semua kolom diproses.

**Hubungan dengan Eliminasi Gauss**
Ini inti metode: eliminasi maju. Operasi `R_i ← R_i − m × R_pivot` adalah operasi baris elementer "menambah kelipatan baris lain", dan tukar baris adalah operasi "menukar dua baris".

**Hasil yang diharapkan untuk contoh ini**
- Kolom 1: pivot = 2.000 pada R1, lalu `R2 <- R2 - (2) x R1` dan `R3 <- R3 - (-1) x R1`. Hasilnya R2 = `[0 1 -5 | 11]` dan R3 = `[0 6 5 | -4]`.
- Kolom 2: pivot = 1.000 pada R2, lalu `R3 <- R3 - (6) x R2`. Hasilnya R3 = `[0 0 35 | -70]`.
- Kolom 3: pivot = 35.000 pada R3. Tidak ada baris di bawahnya, jadi tidak ada operasi.
- Akhir: `Jumlah pivot (rank) = 3` dan matriks segitiga atas `[2 1 3 | 1]`, `[0 1 -5 | 11]`, `[0 0 35 | -70]`.

In [12]:
# CELL 6 - Pemeriksaan jenis solusi dan substitusi balik

# Langkah 1: tentukan jenis solusi dari bentuk akhir matriks
status = None
for i in range(rank, n):                 # baris yang koefisiennya sudah nol semua
    if abs(M[i][n]) > EPSILON:           # bentuk 0 0 0 | c dengan c bukan nol
        status = "tidak_ada"
if status is None:
    status = "tak_hingga" if rank < n else "tunggal"

# Langkah 2: substitusi balik hanya bila solusinya tunggal
solusi = None
if status == "tunggal":
    solusi = [0.0] * n                   # tempat menyimpan nilai t, u, v
    print("=== SUBSTITUSI BALIK ===")
    for i in range(n - 1, -1, -1):       # dari baris terakhir ke baris pertama
        # jumlah = suku-suku yang variabelnya sudah diketahui
        jumlah = sum(M[i][k] * solusi[k] for k in range(i + 1, n))
        solusi[i] = (M[i][n] - jumlah) / M[i][i]
        print(f"  {NAMA_VARIABEL[i]} = ({M[i][n]:.3f} - {jumlah:.3f}) / {M[i][i]:.3f} = {solusi[i]:.6f}")
else:
    print("Substitusi balik tidak dilakukan karena SPL tidak memiliki solusi tunggal.")

=== SUBSTITUSI BALIK ===
  v = (-70.000 - 0.000) / 35.000 = -2.000000
  u = (11.000 - 10.000) / 1.000 = 1.000000
  t = (1.000 - -5.000) / 2.000 = 3.000000


### Penjelasan Cell 6: Pemeriksaan Jenis Solusi dan Substitusi Balik

**Tujuan cell**
1. Menentukan jenis solusi SPL dari bentuk akhir matriks.
2. Jika solusinya tunggal, menghitung nilai v, u, dan t dengan substitusi balik.

**Fungsi variabel**
| Variabel | Fungsi |
|---|---|
| `status` | Jenis solusi: `"tunggal"`, `"tidak_ada"`, atau `"tak_hingga"`. |
| `solusi` | List berisi nilai `[t, u, v]`. Tetap `None` bila solusi tunggal tidak ada. |
| `i` | Nomor baris. Pada substitusi balik bergerak dari 2 turun ke 0. |
| `k` | Nomor kolom variabel yang nilainya sudah diketahui. |
| `jumlah` | Total (koefisien × nilai variabel) untuk variabel yang sudah diketahui, di sebelah kanan pivot. |

**Fungsi tiap bagian kode**
- **Langkah 1:** `for i in range(rank, n)` memeriksa baris-baris yang tidak punya pivot (koefisiennya sudah nol semua). Jika konstanta barisnya bukan nol (`abs(M[i][n]) > EPSILON`), bentuknya `0 0 0 | c` yang berarti `0 = c` (mustahil), sehingga `status = "tidak_ada"`.
- Jika belum ada pertentangan: `rank < n` berarti ada variabel tanpa pivot, sehingga `"tak_hingga"`. Jika `rank == n`, sehingga `"tunggal"`.
- **Langkah 2:** hanya dijalankan bila `status == "tunggal"`. Program **tidak memaksa** menghitung tiga nilai pada kasus lain.
- `solusi = [0.0] * n`: menyiapkan tiga tempat kosong.
- `for i in range(n - 1, -1, -1)`: mulai dari baris terakhir naik ke baris pertama.
- `jumlah = sum(...)`: menghitung suku-suku di kanan pivot yang nilainya sudah diketahui. Pada baris terakhir tidak ada suku yang diketahui, jadi `jumlah = 0`.
- `solusi[i] = (M[i][n] - jumlah) / M[i][i]`: rumus substitusi balik, yaitu (konstanta − jumlah) ÷ pivot.
- `print(...)`: menampilkan perhitungan tiap variabel.

**Hubungan dengan Eliminasi Gauss**
Ini tahap kedua metode, yaitu substitusi balik. Matriks segitiga atas dari Cell 5 membuat baris terakhir hanya memuat v, baris kedua memuat u dan v, dan seterusnya, sehingga variabel bisa dihitung dari bawah ke atas.

**Hasil yang diharapkan untuk contoh ini**

In [13]:
# CELL 7 - Menampilkan hasil akhir

print("=== HASIL AKHIR ===")
if status == "tunggal":
    print("SPL memiliki SOLUSI TUNGGAL:")
    for j in range(n):
        print(f"  {NAMA_VARIABEL[j]} = {solusi[j]:.6f}")
elif status == "tak_hingga":
    print(f"SPL memiliki TAK HINGGA BANYAK SOLUSI (rank = {rank} < {n} variabel).")
    print("Tidak ada satu nilai t, u, v yang bisa ditetapkan sebagai solusi tunggal.")
else:
    print("SPL TIDAK MEMILIKI SOLUSI.")
    print("Terdapat baris berbentuk 0 0 0 | c dengan c bukan nol, yaitu persamaan yang mustahil.")

=== HASIL AKHIR ===
SPL memiliki SOLUSI TUNGGAL:
  t = 3.000000
  u = 1.000000
  v = -2.000000


### Penjelasan Cell 7: Menampilkan Hasil Akhir

**Tujuan cell**
Menyajikan kesimpulan SPL secara ringkas sesuai jenis solusinya.

**Fungsi variabel**
Cell ini tidak membuat variabel baru. Ia memakai `status`, `solusi`, `rank`, `n`, dan `NAMA_VARIABEL` dari cell sebelumnya, serta `j` sebagai penghitung perulangan.

**Fungsi tiap bagian kode**
- `if status == "tunggal":`: mencetak nilai t, u, v dengan 6 angka desimal (`:.6f`) menggunakan perulangan `for j`.
- `elif status == "tak_hingga":`: mencetak bahwa solusinya tak hingga banyak, disertai nilai rank sebagai alasan (rank lebih kecil dari jumlah variabel). Tidak ada nilai t, u, v yang ditampilkan.
- `else:`: kasus `"tidak_ada"`, mencetak bahwa SPL tidak memiliki solusi beserta alasannya.

**Hubungan dengan Eliminasi Gauss**
Cell ini menafsirkan hasil eliminasi: solusi tunggal bila rank = 3, tak hingga solusi bila rank < 3 tanpa pertentangan, dan tidak ada solusi bila muncul baris `0 0 0 | c`.

**Hasil yang diharapkan untuk contoh ini**
```
=== HASIL AKHIR ===
SPL memiliki SOLUSI TUNGGAL:
  t = 3.000000
  u = 1.000000
  v = -2.000000
```

In [14]:
# CELL 8 - Validasi: masukkan kembali nilai t, u, v ke persamaan awal

print("=== VALIDASI ===")
if solusi is None:
    print("Validasi dilewati karena tidak ada solusi tunggal untuk diperiksa.")
else:
    semua_cocok = True
    for i in range(n):
        # ruas kiri persamaan ke-i dihitung dari A asli (bukan matriks hasil eliminasi)
        hasil = sum(A[i][j] * solusi[j] for j in range(n))
        selisih = abs(hasil - b[i])
        cocok = selisih < 1e-6           # toleransi kecil karena pembulatan desimal
        semua_cocok = semua_cocok and cocok
        print(f"  Persamaan {i+1}: ruas kiri = {hasil:.6f}, konstanta = {b[i]}, selisih = {selisih:.2e} -> {'COCOK' if cocok else 'TIDAK COCOK'}")
    print("\nKesimpulan:", "solusi VALID." if semua_cocok else "solusi TIDAK valid, periksa kembali perhitungan.")

=== VALIDASI ===
  Persamaan 1: ruas kiri = 1.000000, konstanta = 1, selisih = 0.00e+00 -> COCOK
  Persamaan 2: ruas kiri = 13.000000, konstanta = 13, selisih = 0.00e+00 -> COCOK
  Persamaan 3: ruas kiri = -5.000000, konstanta = -5, selisih = 0.00e+00 -> COCOK

Kesimpulan: solusi VALID.


### Penjelasan Cell 8: Validasi Hasil

**Tujuan cell**
Memeriksa kebenaran solusi dengan memasukkan kembali nilai t, u, v ke **persamaan awal**.

**Fungsi variabel**
| Variabel | Fungsi |
|---|---|
| `semua_cocok` | Penanda keseluruhan, awalnya `True`. Menjadi `False` jika ada satu persamaan yang tidak cocok. |
| `hasil` | Nilai ruas kiri persamaan ke-i setelah t, u, v dimasukkan. |
| `selisih` | Jarak antara `hasil` dan konstanta `b[i]`. |
| `cocok` | `True` bila `selisih` lebih kecil dari `1e-6`. |

**Fungsi tiap bagian kode**
- `if solusi is None:`: bila tidak ada solusi tunggal, tidak ada yang bisa divalidasi sehingga cell hanya mencetak keterangan.
- `hasil = sum(A[i][j] * solusi[j] ...)`: menghitung `koef_t·t + koef_u·u + koef_v·v` memakai `A` **asli** dari Cell 3. Ini penting agar yang diuji adalah persamaan awal, bukan matriks yang sudah diubah.
- `selisih = abs(hasil - b[i])`: mengukur perbedaan hasil dengan konstanta.
- `cocok = selisih < 1e-6`: memakai toleransi, bukan `==`, karena bilangan desimal komputer tidak selalu tepat.
- `semua_cocok = semua_cocok and cocok`: tetap `True` hanya bila semua persamaan cocok.
- `print(...)` terakhir: menampilkan kesimpulan.

**Hubungan dengan Eliminasi Gauss**
Eliminasi Gauss adalah metode eksak, tetapi komputer dapat menimbulkan galat pembulatan. Validasi memastikan hasil akhir benar-benar memenuhi ketiga persamaan awal.

**Hasil yang diharapkan untuk contoh ini**